# Notebook 13 — ML for Genomics and Sequence Data

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 13 of 15  
**Time estimate:** 90 minutes

> Sequence is the primary data type in computational biology.
> Understanding how to featurize sequences — and the limits of those representations
> — is what separates bioinformatics engineers from ML engineers who happen to touch biology.

---
## Step 1 — Motivation

A transcription factor binds specific DNA sequences. Can we predict whether
a 30-mer sequence will be bound? This is a binary classification problem, but
the input is text (ACGT), not numbers. Featurization — converting sequence to
a numerical vector — is the critical design choice. We examine four approaches:
one-hot encoding, k-mer features, position weight matrices, and a preview of
convolutional approaches (fully developed in Module 14).

---
## Step 2 — Intuition

**One-hot encoding:** Each position in the sequence becomes 4 binary features
(A, C, G, T). A 30-mer becomes a 120-dimensional binary vector.
- Preserves positional information
- Linear models can learn position-specific weights (= a position weight matrix)
- No implicit similarity between nucleotides (unlike embedding)

**k-mer features (bag of words for sequences):**
Count the frequency of all $4^k$ possible k-mers in the sequence.
- $k=2$: 16 features; $k=3$: 64; $k=6$: 4,096
- Captures local sequence composition but loses positional information
- Very fast to compute; works well with linear SVM (state of art for many tasks
  before deep learning)

**Position Weight Matrix (PWM):**
A $4 \times L$ matrix where entry $(b, i)$ is the log-odds of base $b$ at position $i$.
Standard representation of a transcription factor binding motif.
Score of a sequence = sum of log-odds at each position.

**Variant effect prediction:**
Given a reference sequence and a single nucleotide variant (SNV),
predict whether the variant is pathogenic.
Features: conservation, functional annotations, allele frequency.
CADD, PolyPhen-2, SIFT, REVEL all use ML on these features.

---
## Step 3 — Biological Background

**JASPAR motif database:**
Curated database of TF binding motifs as PWMs. Each TF has a 4×L matrix.
FIMO (Find Individual Motif Occurrences) scans sequences for motif matches
using the log-likelihood score.

**ChIP-seq peak classification:**
ChIP-seq identifies genomic regions bound by a TF. Bound peaks (positive)
vs. flanking regions (negative) → binary classification. DeepBind (Alipanahi 2015)
showed that convolutional neural networks on one-hot encoded sequences outperform
PWMs for TF binding prediction.

**Splice site prediction:**
The pre-mRNA sequence at intron-exon boundaries follows a consensus (GT-AG rule).
ML can distinguish true splice sites from cryptic sites using sequence context.
SpliceAI (Jaganathan 2019) uses deep residual networks on 10kb windows.

**CADD score construction:**
Simulated variants (ancestral → derived alleles, presumed neutral) vs.
observed human variants (mix of neutral and deleterious). Features:
60+ annotations including conservation (PhyloP, GERP), regulatory annotations
(DNase, histone marks), coding effect (synonymous/nonsynonymous), splicing.
Model: SVM + GBT ensemble trained on this composite dataset.

---
## Step 4 — Mathematical Explanation

**PWM scoring:**
Given a PWM $M$ of shape $(4, L)$ and a sequence $s$ of length $L$:
$$\text{score}(s, M) = \sum_{i=1}^L M[\text{base}(s_i), i]$$

where $M[b, i] = \log_2 \frac{f_{b,i} + p_b \lambda}{(1+\lambda) p_b}$
($f_{b,i}$ = observed frequency, $p_b$ = background, $\lambda$ = pseudocount)

**k-mer feature vector:**
For a sequence of length $L$, extract all $L - k + 1$ k-mers and count each:
$$\phi_k(s) \in \mathbb{R}^{4^k}, \quad \phi_k(s)[\tau] = \sum_{i=1}^{L-k+1} \mathbf{1}[s_{i:i+k} = \tau]$$

**Spectrum kernel (mismatch kernel):**
Rather than exact k-mer matches, the mismatch kernel counts approximate matches
(allowing $m$ mismatches). $K(s, t) = \phi_k(s) \cdot \phi_k(t)$ with mismatch.
Linear SVM + spectrum kernel was state-of-the-art for TF binding before deep learning.

**Information content of a motif position:**
$$IC_i = \log_2(4) - H_i = 2 - \left(-\sum_{b} f_{b,i} \log_2 f_{b,i}\right)$$

High IC = conserved position (must be correct base for binding).
Low IC = degenerate position (any base is acceptable).

In [ ]:
# Step 6 — Python: Sequence featurization + TF binding prediction
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from itertools import product

# ---- Sequence utilities ----
NUCL = 'ACGT'
NUCL_IDX = {b: i for i, b in enumerate(NUCL)}

def one_hot(seq):
    """Convert DNA string to (4, L) one-hot array."""
    L = len(seq)
    mat = np.zeros((4, L), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in NUCL_IDX:
            mat[NUCL_IDX[b], i] = 1.0
    return mat

def kmer_features(seq, k=3):
    """Extract k-mer frequency vector."""
    kmers = [''.join(p) for p in product(NUCL, repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    vec = np.zeros(len(kmers), dtype=np.float32)
    for i in range(len(seq) - k + 1):
        km = seq[i:i+k]
        if km in kmer_idx:
            vec[kmer_idx[km]] += 1
    return vec / max(len(seq) - k + 1, 1)  # normalize to frequency

def pwm_score(seq, pwm):
    """Score a sequence against a PWM (4 x L)."""
    L = pwm.shape[1]
    assert len(seq) == L, f'Sequence length {len(seq)} != PWM length {L}'
    score = 0.0
    for i, b in enumerate(seq):
        if b in NUCL_IDX:
            score += pwm[NUCL_IDX[b], i]
    return score

# ---- Synthetic TF binding dataset ----
# Simulate a CTCF-like motif: 10-mer with strong conserved positions
rng = np.random.default_rng(42)
MOTIF = 'CCGCGNGGNGG'  # CTCF-like consensus (N = any)
MOTIF_L = 11
SEQ_L = 30

def generate_binding_dataset(n_pos=500, n_neg=500, motif=MOTIF, seq_l=SEQ_L, rng=rng):
    sequences, labels = [], []
    
    # Positive sequences: contain the motif (with ~1 substitution)
    for _ in range(n_pos):
        seq = list(''.join(rng.choice(list(NUCL), seq_l)))
        pos = rng.integers(0, seq_l - MOTIF_L)
        for j, b in enumerate(motif):
            if b == 'N':
                seq[pos + j] = rng.choice(list(NUCL))
            elif rng.random() < 0.1:  # 10% chance of substitution
                seq[pos + j] = rng.choice([x for x in NUCL if x != b])
            else:
                seq[pos + j] = b
        sequences.append(''.join(seq))
        labels.append(1)
    
    # Negative sequences: purely random
    for _ in range(n_neg):
        sequences.append(''.join(rng.choice(list(NUCL), seq_l)))
        labels.append(0)
    
    return sequences, np.array(labels)

sequences, y_bind = generate_binding_dataset()
print(f'Binding dataset: {len(sequences)} sequences ({y_bind.sum()} bound, {(1-y_bind).sum()} unbound)')
print(f'Example positive: {sequences[0]}')
print(f'Example negative: {sequences[500]}')

# ---- Build feature matrices ----
# One-hot (flattened)
X_onehot = np.array([one_hot(s).ravel() for s in sequences])
print(f'\nOne-hot shape: {X_onehot.shape}')

# k-mer frequencies (k=3 and k=4)
X_kmer3 = np.array([kmer_features(s, k=3) for s in sequences])
X_kmer4 = np.array([kmer_features(s, k=4) for s in sequences])
print(f'k=3 features shape: {X_kmer3.shape}')
print(f'k=4 features shape: {X_kmer4.shape}')

# PWM: learn from positive sequences
def learn_pwm(sequences, labels, motif_l, rng, n_samples=100, pseudocount=0.5):
    """Estimate a PWM from positive sequences (simple: use motif region)."""
    counts = np.ones((4, motif_l)) * pseudocount  # pseudocounts
    n_pos = 0
    for seq, lbl in zip(sequences, labels):
        if lbl == 1 and n_pos < n_samples:
            # Scan for best motif position
            best_pos, best_score = 0, -np.inf
            for p in range(len(seq) - motif_l + 1):
                sc = sum(1 for j, b in enumerate(seq[p:p+motif_l]) if b == MOTIF[j] or MOTIF[j] == 'N')
                if sc > best_score:
                    best_score, best_pos = sc, p
            for j, b in enumerate(seq[best_pos:best_pos+motif_l]):
                if b in NUCL_IDX:
                    counts[NUCL_IDX[b], j] += 1
            n_pos += 1
    freq = counts / counts.sum(axis=0, keepdims=True)
    bg = 0.25  # uniform background
    pwm = np.log2(freq / bg)
    return pwm

pwm = learn_pwm(sequences, y_bind, MOTIF_L, rng)
print(f'\nPWM shape: {pwm.shape}')

# Score all sequences with PWM (sliding window, take max)
def max_pwm_score(seq, pwm):
    L = pwm.shape[1]
    scores = [pwm_score(seq[i:i+L], pwm) for i in range(len(seq)-L+1)]
    return max(scores) if scores else 0

X_pwm = np.array([max_pwm_score(s, pwm) for s in sequences]).reshape(-1, 1)
print(f'PWM scores shape: {X_pwm.shape}')

# ---- Evaluate models ----
cv = StratifiedKFold(5, shuffle=True, random_state=42)
results = {}
for name, X_feat in [
    ('One-hot + LR', X_onehot),
    ('k=3 + LR', X_kmer3),
    ('k=4 + SVM (linear)', X_kmer4),
    ('PWM score + LR', X_pwm),
]:
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_feat)
    if 'SVM' in name:
        clf = SVC(kernel='linear', C=1, probability=True)
    else:
        clf = LogisticRegression(C=1, max_iter=5000)
    auc = cross_val_score(clf, X_sc, y_bind, cv=cv, scoring='roc_auc')
    results[name] = auc
    print(f'{name}: AUROC={auc.mean():.3f}±{auc.std():.3f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: PWM logo (information content)
ax = axes[0]
ic = np.zeros(MOTIF_L)
bg = 0.25
for i in range(MOTIF_L):
    freq_col = (np.maximum(2**pwm[:, i] * bg, 1e-8))
    freq_col = freq_col / freq_col.sum()
    entropy = -np.sum(freq_col * np.log2(np.maximum(freq_col, 1e-8)))
    ic[i] = 2 - entropy

for i in range(MOTIF_L):
    freq_col = (np.maximum(2**pwm[:, i] * bg, 1e-8))
    freq_col = freq_col / freq_col.sum()
    order = np.argsort(freq_col)
    bottom = 0
    colors_nucl = ['#00a651', '#3953a4', '#fdb813', '#ed1c24']  # A=green,C=blue,G=yellow,T=red
    for b in order:
        height = freq_col[b] * ic[i]
        ax.bar(i, height, bottom=bottom, color=colors_nucl[b],
                 edgecolor='none', width=0.8)
        bottom += height
ax.set_xlabel('Motif position')
ax.set_ylabel('Information content (bits)')
ax.set_title(f'A. PWM sequence logo\n(learned from positive sequences)')
ax.set_xticks(range(MOTIF_L))
from matplotlib.patches import Patch
leg = [Patch(color=c, label=b) for b, c in zip(NUCL, ['#00a651','#3953a4','#fdb813','#ed1c24'])]
ax.legend(handles=leg, fontsize=8, loc='upper right')

# Panel B: PWM score distributions (positive vs. negative)
ax = axes[1]
scores_pos = X_pwm[y_bind == 1].ravel()
scores_neg = X_pwm[y_bind == 0].ravel()
bins = np.linspace(X_pwm.min(), X_pwm.max(), 40)
ax.hist(scores_pos, bins=bins, alpha=0.6, color='tomato', label='Bound')
ax.hist(scores_neg, bins=bins, alpha=0.6, color='steelblue', label='Unbound')
ax.set_xlabel('Max PWM score')
ax.set_ylabel('Count')
ax.set_title('B. PWM score distributions\n(bound sequences score higher)')
ax.legend(fontsize=9)

# Panel C: Model AUROC comparison
ax = axes[2]
names = list(results.keys())
means = [results[n].mean() for n in names]
stds = [results[n].std() for n in names]
colors_bar = ['steelblue', 'green', 'orange', 'tomato']
ax.barh(range(len(names)), means, xerr=stds, capsize=4,
          color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.axvline(0.5, color='grey', ls='--', lw=0.8, label='Random')
ax.set_xlabel('5-fold CV AUROC')
ax.set_title('C. Feature representations\n(k-mer + SVM competitive with one-hot)')
ax.set_xlim(0.4, 1.0)
ax.legend(fontsize=8)

# Panel D: k-mer importance (top 20 3-mers by LR coefficient)
from sklearn.linear_model import LogisticRegression as LR
clf_kmer = LR(C=1, max_iter=5000)
clf_kmer.fit(StandardScaler().fit_transform(X_kmer3), y_bind)
ax = axes[3]
kmers_list = [''.join(p) for p in product(NUCL, repeat=3)]
coef = clf_kmer.coef_.ravel()
top20_kmer = np.argsort(np.abs(coef))[-20:][::-1]
ax.barh(range(20), coef[top20_kmer],
          color=['tomato' if c > 0 else 'steelblue' for c in coef[top20_kmer]],
          edgecolor='black', alpha=0.8)
ax.set_yticks(range(20))
ax.set_yticklabels([kmers_list[i] for i in top20_kmer], fontsize=8)
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('LR coefficient (3-mer)')
ax.set_title('D. Top 3-mers by LR weight\n(positive = enriched in bound sequences)')

plt.suptitle('Module 13 NB13: ML for Genomics and Sequence Data', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ml_genomics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the information content calculation for each PWM position from scratch.
   Verify your values match the logo heights in Panel A.
2. Compare k-mer sizes $k = 2, 3, 4, 5$. Plot AUROC vs. $k$. Is there a sweet spot?
   (Larger $k$ captures longer motifs but creates sparser feature matrices.)
3. Implement a simple sliding window one-hot classifier that averages one-hot
   features across a window, rather than concatenating positionally. Compare to
   concatenation. What information is lost?
4. Download a real JASPAR motif (e.g., CTCF, GATA1) and score the synthetic
   sequences against it. How well does the canonical motif score on your simulated data?

---
## Step 10 — Quiz

1. Write the PWM scoring formula. What does each entry represent?
2. What is a k-mer feature vector? What information is lost vs. one-hot encoding?
3. What is information content at a motif position? What does IC = 2 bits mean?
4. What is the challenge of variant effect prediction that makes it fundamentally
   different from standard binary classification?
5. Why did DeepBind outperform PWM-based methods? What architectural feature enabled this?

---
## Step 12 — Reflection

> *[In one paragraph: explain why k-mer features trained with an SVM can be seen
> as approximating what a convolutional neural network learns, and what
> advantage the CNN has over k-mer + SVM for TF binding prediction.]*

---
*Next: `14_ml_for_clinical_omics.ipynb`*